# Outline

1. Required Installation
2. QR-PUF
    - 2.1 BB84 Qubit Encoding
3. XOR-APUF (CPUF) Implementation
4. BB84-Encoded PUF Responses
    - 4.1 Threat Model
    - 4.2 From One-Copy Quantum Measurement to Label Noise
5. Modelling Attack (`LRAttack2021`) for HPUF
6. Runtime Patches for `LRAttack2021`
7. Bonus: Qiskit Implementation
    - 7.1 Setup
    - 7.2 BB84 State-Preparation Circuits
    - 7.3 Eve's Optimal Measurement — the Breidbart Basis
    - 7.4 Running the Circuit Simulation and Verifying the Analytical Model
    - 7.5 Modelling Attack (`LRAttack2021`) on the Qiskit-Simulated HPUF
    - 7.6 Runtime Patches and the Attack Helper
    - 7.7 Running the Attack

## 1. Required Installation

Run the cell below once to install the required packages. (Drop `--break-system-packages` if you're
running inside a virtual environment — it's only needed on systems with an "externally managed"
Python installation, e.g. recent Debian/Ubuntu.)

In [ ]:
%pip install pypuf==3.2.1
%pip install tensorflow
%pip install "numpy<2.0" 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pypuf 3.2.1 requires tensorflow~=2.4.0, but you have tensorflow 2.21.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Traceback (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_mod

## 2. QR-PUF

As we saw in Lab 1, XOR Arbiter PUFs (A-PUFs) are physically unclonable but **mathematically
modelable**: given enough challenge-response pairs (CRPs), an attacker (Eve) can clone the PUF's
behavior with a simple logistic-regression model (~98-99% accuracy), because the additive delay
model is **linearly separable** under the $\phi$-parity transform.

This raises a question: *what if the response isn't handed to the attacker as an observable
classical bit?*

The **Quantum-Readout PUF (QR-PUF)**, proposed by Škorić, keeps the PUF itself classical but carries
the challenge/response over **quantum states**. The **HPUF (Hybrid Locked PUF)** construction used in
this lab is a concrete instance:

$$
\text{Classical PUF output} \;\xrightarrow{\text{BB84 encoding}}\; \text{Non-orthogonal quantum state}
$$

Two properties of quantum mechanics make this useful:

1. **No-cloning theorem** — an unknown quantum state can't be perfectly copied, so Eve can't both
   forward the qubit to the verifier and keep a copy for her own analysis.
2. **Indistinguishability of non-orthogonal states** — if the encoding basis ($Z$ or $X$) is randomly
   chosen, measuring a single copy in the wrong basis returns a **completely random bit**, unrelated
   to the original value.

### 2.1 BB84 Qubit Encoding

BB84 (Bennett & Brassard, 1984) is normally a key-distribution protocol; here we only use its
**encoding scheme** — turning a classical bit pair into an unclonable quantum state.

| Basis | Symbol | 0 → State | 1 → State |
|---|---|---|---|
| Rectilinear ($Z$) | $\{\lvert 0\rangle, \lvert 1\rangle\}$ | $\lvert 0\rangle$ | $\lvert 1\rangle$ |
| Diagonal ($X$) | $\{\lvert +\rangle, \lvert -\rangle\}$ | $\lvert +\rangle = \frac{\lvert 0\rangle + \lvert 1\rangle}{\sqrt{2}}$ | $\lvert -\rangle = \frac{\lvert 0\rangle - \lvert 1\rangle}{\sqrt{2}}$ |

These four states are **non-orthogonal** across the two bases — the cornerstone of the scheme's
security. Each qubit carries two classical bits:

- **basis bit** — which basis was used (0 → $Z$, 1 → $X$)
- **value bit** — which state within that basis (0 or 1)

The PUF's (basis, value) pairs map to a qubit as:

$$
\lvert \psi \rangle =
\begin{cases}
\lvert 0 \rangle & \text{basis}=0,\ \text{value}=0 \\
\lvert 1 \rangle & \text{basis}=0,\ \text{value}=1 \\
\lvert + \rangle & \text{basis}=1,\ \text{value}=0 \\
\lvert - \rangle & \text{basis}=1,\ \text{value}=1
\end{cases}
$$

The preparation basis is physically invisible to whoever receives the qubit. Measuring in the
**correct** basis recovers the value bit with certainty; measuring in the **wrong** basis gives a
uniformly random bit, carrying no information about the original value.

We verify this later by simulating Eve's measurement. A naive strategy — guess the basis, then
measure — gets the basis right only half the time, and a wrong-basis measurement is fully random, so
it flips the true response on average **25%** of the time. Section 4.2 derives the *optimal*
single-copy strategy, which does better (~14.64% error).

## 3. XOR-APUF (CPUF) Implementation

The following code implements the XOR-APUF used as the CPUF in this lab (see Lab 1 for details):

In [1]:
import numpy as np
from pypuf.simulation import XORArbiterPUF
from pypuf.io import random_inputs

n = 64
k = 5
puf = XORArbiterPUF(n=n, k=k, seed=1, noisiness=0.0)

train_N = 200000
test_N  = train_N - (int)(train_N * 0.8)
train_challenges = random_inputs(n=n, N=train_N, seed=2)
train_responses = puf.eval(train_challenges)    
test_challenges = random_inputs(n=n, N=test_N, seed=2)
test_responses = puf.eval(test_challenges)  

## 4. BB84-Encoded PUF Responses

For each challenge $c_i$, the CPUF produces a basis bit $b_i=f_b(c_i)$ and a value bit
$\theta_i=f_\theta(c_i)$; the HPUF would encode these as the BB84 state $|\psi_{(b_i,\theta_i)}\rangle$.

### 4.1 Threat Model

The adversary receives $D_Q = \{(c_i, |\psi_{(b_i,\theta_i)}\rangle)\}_{i=1}^{q}$, and:

- knows the PUF architecture and the BB84 encoding rule, but not the hidden parameters of $f_b$/$f_\theta$;
- gets exactly **one** copy of each quantum response — no repeated measurements of the same state;
- has arbitrary (single-copy) measurement capability;
- wants a classical ML model that predicts the true response bit on unseen challenges.

The one-copy restriction is essential: with multiple copies of the same state, the adversary could
improve discrimination beyond the fixed error rate derived next.

### 4.2 From One-Copy Quantum Measurement to Label Noise

If $b=0$, the emitted state is $|0\rangle$ or $|+\rangle$ with equal probability; if $b=1$, it's
$|1\rangle$ or $|-\rangle$. The adversary must distinguish the two mixed states:

$$
\rho_0 = \tfrac12\big(|0\rangle\langle 0| + |+\rangle\langle +|\big), \qquad
\rho_1 = \tfrac12\big(|1\rangle\langle 1| + |-\rangle\langle -|\big)
$$

Optimal single-copy discrimination of these gives:

$$
p_{guess} = \tfrac12 + \tfrac{1}{2\sqrt{2}} \approx 0.853553, \qquad
p_{error} = 1 - p_{guess} \approx 0.146447
$$

This ~14.64% isn't physical noise from the PUF or from BB84 preparation — it's the effective
classical label-error rate after the *optimal* one-copy measurement. So instead of simulating the
measurement itself, we can generate HPUF responses directly from this known statistic:

$$
b \rightarrow |\psi\rangle \rightarrow \text{optimal measurement} \rightarrow \tilde{b}, \qquad
\Pr[\tilde{b} = b] = 0.853553,\ \ \Pr[\tilde{b} \neq b] = 0.146447
$$

In [2]:
clean_responses = train_responses.copy()

p_guess = 0.5 + 1/(2*np.sqrt(2))
p_error = 1 - p_guess
print('p_guess =', p_guess)
print('p_error =', p_error)

rng = np.random.default_rng(12345)
measured_responses = clean_responses.copy()
flip = rng.random(len(measured_responses)) < p_error
measured_responses[flip] *= -1

observed_error = np.mean(
    measured_responses.reshape(-1) != clean_responses.reshape(-1)
)
print('Observed extraction error:', observed_error)

p_guess = 0.8535533905932737
p_error = 0.14644660940672627
Observed extraction error: 0.14664


For a large training database, the observed extraction error should be close to 0.1464.

## 5. Modelling Attack (`LRAttack2021`) for HPUF

In [5]:
import matplotlib.pyplot as plt
import tensorflow as tf
from pypuf.simulation.base import LTFArray
from pypuf.io import random_inputs, ChallengeResponseSet
from pypuf.attack import LRAttack2021
import pypuf.attack

quantum_crps = ChallengeResponseSet(
    train_challenges[:train_N],
    measured_responses[:train_N]
)
attack_q = LRAttack2021(
    quantum_crps,
    seed=100,
    k=k,
    bs=1000,
    lr=0.001,
    epochs=100,
    stop_validation_accuracy=1.0
)
model_q = attack_q.fit()

predicted_q = model_q.eval(test_challenges)
accuracy_q = np.mean(
    predicted_q.reshape(-1) == test_responses.reshape(-1)
)
print('BB84-interface attack accuracy:', accuracy_q)

Epoch 1/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5002 - loss: 0.6932 - val_accuracy: 0.4935 - val_loss: 0.6931
Epoch 2/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step - accuracy: 0.5014 - loss: 0.6931 - val_accuracy: 0.5095 - val_loss: 0.6931
Epoch 3/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 975us/step - accuracy: 0.5038 - loss: 0.6929 - val_accuracy: 0.5130 - val_loss: 0.6930
Epoch 4/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 971us/step - accuracy: 0.5069 - loss: 0.6926 - val_accuracy: 0.5065 - val_loss: 0.6932
Epoch 5/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 976us/step - accuracy: 0.5069 - loss: 0.6924 - val_accuracy: 0.5065 - val_loss: 0.6933
Epoch 6/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5087 - loss: 0.6923 - val_accuracy: 0.5175 - val_loss: 0.6934
Epoch 7/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 988us/step - accuracy: 0.5106 - loss: 0.6922 - val_accuracy: 0.5035 - val_loss: 0.6935
Epoch 8/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5103 - loss: 0.6921 

## 6. Runtime Patches for `LRAttack2021`

If the previous cell raised:

```
Traceback (most recent call last):
  File "../QPUF Tutorial/deneme.py", line 27, in <module>
    model = attack.fit()
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 145, in fit
    self._history = model.fit(
  File "../.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "../.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 89, in accuracy
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)
TypeError: Expected int8, but got 0.5 of type 'float'.
```

run the cell below, then re-run the attack cell above.

In [4]:
def keras_to_pypuf_fixed(self, keras_model):
    n, k = self.crps.challenges.shape[1], self.k
    weights = np.zeros(shape=(k, n))
    bias = np.zeros(shape=(k,))
    for l in range(k):
        layer_weights = keras_model.layers[l + 1].get_weights()
        weights[l] = layer_weights[0][:, 0]
        bias[l] = layer_weights[1][0]
    return LTFArray(weight_array=weights, bias=bias, transform=XORArbiterPUF.transform_atf)
pypuf.attack.LRAttack2021.keras_to_pypuf = keras_to_pypuf_fixed

@staticmethod
def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.losses.binary_crossentropy(.5 - .5 * y_true, .5 - .5 * y_pred)

@staticmethod
def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)

pypuf.attack.LRAttack2021.loss = loss
pypuf.attack.LRAttack2021.accuracy = accuracy

## 7. Bonus: Qiskit Implementation

Sections 4-6 modelled the BB84 measurement **analytically**: instead of preparing and measuring real
qubits, we flipped each response bit with probability $p_{error}$. That shortcut is exact in
expectation, but this section builds the real thing — single-qubit BB84 circuits, Eve's optimal
measurement on Qiskit's `AerSimulator`, and a check that the empirical bit-flip rate matches
$p_{error}\approx 0.1464$.

It's self-contained and doesn't change anything computed above; it can be run independently once the
cells above have run.

### 7.1 Setup

Qiskit and its Aer simulator backend are not required by the rest of the notebook, so they are
installed separately here. `qiskit`/`qiskit-aer` can pull in a different `numpy`/`scipy` than the
`pypuf`/`tensorflow` stack installed above expects, so the second line reinstalls `numpy` and `scipy`
afterward to keep both stacks working in the same kernel.

In [ ]:
%pip install "qiskit" "qiskit-aer" 
%pip install "numpy" "scipy" 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 7.2 BB84 State-Preparation Circuits

Recall the encoding from Section 4: a **value bit** $\theta \in \{0,1\}$ (the PUF response bit we
want to protect) is encoded together with a **basis bit** $b\in\{0,1\}$ into one of four
non-orthogonal single-qubit states. Both bits are PUF-derived ($b_i = f_b(c_i)$,
$\theta_i = f_\theta(c_i)$): $\theta$ comes from the CPUF whose responses we're protecting, and $b$
comes from a second, independent CPUF instance playing the role of $f_b$. Using a second PUF (rather
than a public coin flip) means the basis itself is also a PUF secret — an adversary can't query it
directly, only infer it (imperfectly) from the qubit it helped prepare.

| basis $b$ | value $\theta$ | state |
|---|---|---|
| 0 (rectilinear) | 0 | $\lvert 0\rangle$ |
| 0 (rectilinear) | 1 | $\lvert 1\rangle$ |
| 1 (diagonal)    | 0 | $\lvert +\rangle$ |
| 1 (diagonal)    | 1 | $\lvert -\rangle$ |

As a circuit, starting from $\lvert 0\rangle$: apply `X` if $\theta=1$ (selects $\lvert1\rangle$
over $\lvert0\rangle$), then apply `H` if $b=1$ (rotates into the conjugate
$\{\lvert+\rangle,\lvert-\rangle\}$ basis). The basis bit isn't something Eve can read off the
qubit itself — it only determines which state gets prepared.

In [7]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

SIM = AerSimulator()

def build_bb84_circuits(basis_bits, value_bits):
    """One 1-qubit state-preparation circuit per (basis, value) pair, following the
    encoding table above: X if value=1, then H if basis=1."""
    circuits = []
    for basis, value in zip(basis_bits, value_bits):
        qc = QuantumCircuit(1, 1)
        if value == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

### 7.3 Eve's Optimal Measurement — the Breidbart Basis

Eve doesn't know $b$, and gets only a **single copy** of each qubit. The best she can do is measure
in a *fixed* basis that maximizes her average chance of recovering $\theta$, regardless of which
encoding basis was actually used. That optimal fixed basis is the **Breidbart basis** — exactly
halfway between rectilinear ($Z$) and diagonal ($X$), at $22.5^\circ = \pi/8$.

We implement it by rotating the qubit with `RY(-\pi/4)` (mapping the Breidbart basis onto the
computational $Z$ axis) and measuring in the computational basis. This reproduces the closed-form
result from Section 4.2:

$$
p_{guess} = \cos^2(\pi/8) = \tfrac12 + \tfrac{1}{2\sqrt2} \approx 0.853553, \qquad
p_{error} = \sin^2(\pi/8) = \tfrac12 - \tfrac{1}{2\sqrt2} \approx 0.146447.
$$

In [8]:
def eve_breidbart_measurement(circuits):
    measured = []
    for qc in circuits:
        c = qc.copy()
        c.ry(-np.pi / 4, 0)
        c.measure(0, 0)
        measured.append(c)
    result = SIM.run(measured, shots=1, memory=True).result()
    guesses = np.array([int(result.get_memory(i)[0]) for i in range(len(circuits))])
    return guesses

### 7.4 Running the Circuit Simulation and Verifying the Analytical Model

We take the CPUF's clean training responses (`clean_responses`) as the value bits
$\theta_i=f_\theta(c_i)$, and derive the basis bits $b_i=f_b(c_i)$ from a second, independent
`XORArbiterPUF` instance (`puf_basis`) queried on the same challenges — matching the $f_b$/$f_\theta$
model from Section 4, rather than a public coin flip. We build the corresponding BB84 circuits, run
Eve's Breidbart-basis measurement through `AerSimulator` over the full training set, and compare her
empirical error rate against the theoretical $p_{error}\approx 0.1464$. The resulting database
(`measured_responses_qiskit`) is reused below for the `LRAttack2021` attack, so the circuits are only
simulated once.

In [9]:
puf_basis = XORArbiterPUF(n=n, k=k, seed=99, noisiness=0.0)  # f_b: a second, independent CPUF instance

value_bits = ((1 - clean_responses.reshape(-1)) // 2).astype(int)                       # theta_i = f_theta(c_i)
basis_bits = ((1 - puf_basis.eval(train_challenges).reshape(-1)) // 2).astype(int)       # b_i     = f_b(c_i)

circuits = build_bb84_circuits(basis_bits, value_bits)
eve_guess = eve_breidbart_measurement(circuits)

empirical_error = np.mean(eve_guess != value_bits)
print(f'Theoretical p_error:  {p_error:.6f}')
print(f'Empirical p_error:    {empirical_error:.6f}   (N={train_N} simulated qubits)')

# back to pypuf's {-1,1} bipolar convention — this is Eve's attack database, reused below
measured_responses_qiskit = (1 - 2 * eve_guess).reshape(clean_responses.shape).astype(clean_responses.dtype)

Theoretical p_error:  0.146447
Empirical p_error:    0.145865   (N=200000 simulated qubits)


**Takeaway.** The circuit-simulated bit-flip rate matches the closed-form $p_{error}\approx 0.1464$
used to build `measured_responses` earlier in the notebook, within the expected statistical
fluctuation for this sample size. This confirms the analytical shortcut used above is not an
approximation of convenience — it is exactly what an actual BB84 state-preparation-and-measurement
circuit produces under Eve's optimal single-copy strategy.

### 7.5 Modelling Attack (`LRAttack2021`) on the Qiskit-Simulated HPUF

Section 7.4 confirmed the *statistics* of the analytical shortcut match a real BB84 circuit. Here we
go one step further: attack the full quantum-circuit-derived noisy database (one real Qiskit
state-preparation-and-measurement per training challenge, not a Bernoulli coin flip) with the same
`LRAttack2021` attack used in Section 5. This is the closest this notebook gets to attacking an
actual BB84-encoded HPUF end-to-end: PUF $\to$ qubit $\to$ Eve's Breidbart-basis measurement
$\to$ classical ML attack.

### 7.6 Runtime Patches and the Attack Helper

`LRAttack2021` needs the same two patches applied in Section 6 — they're re-applied here so this
section can be run on its own. `run_lrattack_with_restarts` fits the attack once for a given seed,
clearing the Keras session first so repeated calls in this notebook don't accumulate memory.

Note: modelling a $k$-XOR Arbiter PUF is a non-convex problem, so a single seed can occasionally
converge to a poor local optimum (near-chance accuracy) instead of a good one. If a run looks stuck,
re-running the cell with a different `base_seed` is the simplest fix.

In [10]:
def run_lrattack_with_restarts(crps, k, base_seed=100, bs=1000, lr=0.001,
                                epochs=100, stop_validation_accuracy=1.0):
    """Fit LRAttack2021 once with the given seed. Clears the Keras session first so repeated
    calls in this notebook don't accumulate memory (see the note above on non-convex seeds)."""
    tf.keras.backend.clear_session()
    attack = LRAttack2021(
        crps, seed=base_seed, k=k, bs=bs, lr=lr, epochs=epochs,
        stop_validation_accuracy=stop_validation_accuracy,
    )
    model = attack.fit()
    val_acc = attack.history['val_accuracy'][-1]
    print(f'final val_accuracy={val_acc:.4f}')
    return model, attack

### 7.7 Running the Attack

`measured_responses_qiskit` holds Eve's classical database as it actually came out of the Qiskit
circuit simulation. We train `LRAttack2021` on it, then evaluate the resulting model on
`test_challenges` — comparing its predictions against `test_responses`, the CPUF's true, noise-free
response, not a second noisy quantum measurement of the test challenges.

This is the right comparison: what Eve needs to forge is the same value $\theta=f_\theta(c)$ that a
legitimate verifier recovers deterministically once the correct basis is known — that's the whole
point of the encoding, only an adversary *without* the basis suffers the $p_{error}\approx0.1464$
measurement noise. Scoring Eve's clone against a second noisy measurement instead would measure how
well the model fits Eve's own measurement noise, not whether the clone can actually pass
verification.

In [11]:
crps_qiskit = ChallengeResponseSet(train_challenges, measured_responses_qiskit)

print('Qiskit-circuit HPUF attack:')
model_qiskit, attack_qiskit = run_lrattack_with_restarts(crps_qiskit, k=k)

predicted_qiskit = model_qiskit.eval(test_challenges)
accuracy_qiskit = np.mean(
    predicted_qiskit.reshape(-1) == test_responses.reshape(-1)
)
print('Qiskit-circuit HPUF attack accuracy:', accuracy_qiskit)

Qiskit-circuit HPUF attack:
Epoch 1/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.4993 - loss: 0.6932 - val_accuracy: 0.4965 - val_loss: 0.6932
Epoch 2/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 987us/step - accuracy: 0.5034 - loss: 0.6931 - val_accuracy: 0.5005 - val_loss: 0.6932
Epoch 3/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5061 - loss: 0.6929 - val_accuracy: 0.5040 - val_loss: 0.6934
Epoch 4/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5084 - loss: 0.6927 - val_accuracy: 0.5065 - val_loss: 0.6935
Epoch 5/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5091 - loss: 0.6925 - val_accuracy: 0.5020 - val_loss: 0.6936
Epoch 6/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5091 - loss: 0.6923 - val_accuracy: 0.4935 - val_loss: 0.6936
Epoch 7/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5100 - loss: 0.6922 - val_accuracy: 0.5015 - val_loss: 0.6935
Epoch 8/100
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.

**Takeaway.** This accuracy should land close to the `BB84-interface attack accuracy` computed
earlier in the notebook from the analytical shortcut — the two databases are statistically
equivalent (same $p_{error}\approx0.1464$ label-noise process), just produced by different means:
one from a closed-form coin flip, this one from an actual simulated BB84 state preparation and
Breidbart-basis measurement circuit run through Qiskit's `AerSimulator`.